# Day 8: Agents - Letting the Model Decide What to Do Next

**LangChain Hands-on Series | Day 8 of 10**
Trainer: Ajeetkumar

---

Yesterday you wrote a loop by hand: call the model, run the tools it asks for, send back the results, repeat until it stops asking. Congratulations - you already built an agent. You just did not call it that.

An **agent** is exactly that loop: an LLM that, at each step, decides whether to use a tool (and which one) or to give the final answer.

```
               +-----------------------------------------+
               |                                         |
               v                                         |
  question --> MODEL --- wants a tool? --yes--> TOOLS ---+
                  |
                  no
                  |
                  v
             final answer
```

The difference between a **chain** (Days 3 and 6) and an **agent**:

| Chain | Agent |
|-------|-------|
| *You* decide the steps in advance | The *model* decides the steps at runtime |
| Same path every time | Path depends on the question |
| Predictable, cheaper | Flexible, can solve open-ended tasks |
| Great for: RAG, extraction, pipelines | Great for: assistants, multi-step tasks, using many tools |

LangChain's `create_agent` gives you a production-ready version of yesterday's loop, plus things that are hard to build yourself: memory across turns, streaming every step, structured final answers, human approval before risky actions, and **middleware** to customise each step.

Today's plan:

1. Your first agent with `create_agent`
2. Watching it think: streaming the steps
3. Memory with a checkpointer and `thread_id`
4. Structured final answers with `response_format`
5. Passing user information to tools (runtime context)
6. Middleware: dynamic prompts, logging, limits, summarisation
7. Human-in-the-loop: approving a tool call before it runs
8. **Mini project:** a NovaTech HR agent that combines RAG, data lookups and memory

## Setup

In [1]:
import os
import warnings
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

warnings.filterwarnings("ignore")
load_dotenv()
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
print("Ready. Model:", MODEL_NAME)

Ready. Model: gpt-4.1-nano


Some tools to work with - the same style as Day 7.

In [2]:
import re
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a maths expression like '1200 * 12' or '(450 + 75) / 3'."""
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression):
        return "Error: only numbers and + - * / ( ) % are allowed."
    return str(eval(expression))

WEATHER = {"pune": "29 C, partly cloudy", "mumbai": "31 C, humid", "delhi": "34 C, sunny", "bengaluru": "24 C, light rain"}

@tool
def get_weather(city: str) -> str:
    """Get the current weather for an Indian city (demo data)."""
    return WEATHER.get(city.lower(), f"No weather data for {city}.")

## 1. Your first agent

One function call. Give it a model, a list of tools and a system prompt.

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[calculator, get_weather],
    system_prompt="You are a helpful assistant. Use tools when they help. Keep answers short.",
)

Behind the scenes `create_agent` builds a small graph (using LangGraph). Let's draw it - you will recognise yesterday's loop: the model node, the tools node, and the arrow going back.

In [4]:
print(agent.get_graph().draw_ascii())

ImportError: Install grandalf to draw graphs: `pip install grandalf`.

An agent takes a dictionary with a list of `messages` and returns the full updated list.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Pune?"}]})

print(result["messages"][-1].content)      # the last message is the final answer

**Pause and predict:** how many messages do you think are in `result["messages"]`? Count the steps: question, ...

In [ ]:
for m in result["messages"]:
    m.pretty_print()

<details>
<summary>Click to see the explanation</summary>

Usually four: your **human** question, an **AI** message asking for `get_weather`, a **tool** message with the weather, and the final **AI** answer. Exactly the same sequence you built by hand on Day 7 - now the agent manages it for you.
</details>

## 2. Watching the agent work: streaming

For anything beyond a demo you want to *see* what the agent is doing - for debugging, and for showing "Checking the weather..." in a user interface. `stream_mode="updates"` gives you one update each time a step finishes.

In [ ]:
question = {"messages": [{"role": "user", "content": "What is 1250 * 12, and what's the weather in Delhi?"}]}

for update in agent.stream(question, stream_mode="updates"):
    for step_name, data in update.items():
        last = data["messages"][-1]
        if getattr(last, "tool_calls", None):
            print(f"[{step_name}] wants to call:", [(c["name"], c["args"]) for c in last.tool_calls])
        else:
            print(f"[{step_name}] {last.type}: {last.content}")

`stream_mode="messages"` streams the **words** of the answer as they are generated - the ChatGPT typing effect, but for agents.

In [ ]:
for token, meta in agent.stream(
    {"messages": [{"role": "user", "content": "Give me 3 tips for travelling in Mumbai in one short line each."}]},
    stream_mode="messages",
):
    if meta["langgraph_node"] == "model" and token.content:     # only text from the model, not tool output
        print(token.content, end="", flush=True)

## 3. Memory: remembering earlier turns

Remember Day 4's `session_id`? Agents use the same idea with two new names:

- a **checkpointer** - where conversations are saved (`InMemorySaver` for learning; Postgres or SQLite savers for production)
- a **thread_id** - which conversation this is

```
 thread "asha"  --> [ all of Asha's messages ]
 thread "rahul" --> [ all of Rahul's messages ]
```

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=llm,
    tools=[calculator, get_weather],
    system_prompt="You are a helpful assistant. Keep answers short.",
    checkpointer=InMemorySaver(),          # saves the conversation after every step
)

asha = {"configurable": {"thread_id": "asha"}}

memory_agent.invoke({"messages": [{"role": "user", "content": "Hi, I'm Asha. I live in Bengaluru."}]}, asha)
reply = memory_agent.invoke({"messages": [{"role": "user", "content": "How's the weather where I live?"}]}, asha)

print(reply["messages"][-1].content)

Notice we only sent the **new** message the second time. The checkpointer loaded the earlier ones for us. A different thread starts fresh:

In [ ]:
other = {"configurable": {"thread_id": "someone_else"}}
print(memory_agent.invoke({"messages": [{"role": "user", "content": "Where do I live?"}]}, other)["messages"][-1].content)

In [ ]:
saved = memory_agent.get_state(asha)                  # peek at what is stored for Asha's thread
print("Messages stored for Asha:", len(saved.values["messages"]))

## 4. Structured final answers

Just like `with_structured_output` on Day 2, you can ask the agent to finish with a Pydantic object instead of free text. Very useful when another program consumes the agent's result.

In [ ]:
from pydantic import BaseModel, Field

class TripAdvice(BaseModel):
    city: str = Field(description="The city")
    weather: str = Field(description="Current weather")
    carry_umbrella: bool = Field(description="True if rain is likely")
    tip: str = Field(description="One short travel tip")

advice_agent = create_agent(model=llm, tools=[get_weather], response_format=TripAdvice)

result = advice_agent.invoke({"messages": [{"role": "user", "content": "I'm going to Bengaluru today. Any advice?"}]})
result["structured_response"]

## 5. Giving tools information about the user (runtime context)

In a real app, a tool often needs to know *who* is asking - for example, to look up *their* leave balance. You should **not** rely on the model to pass the user's ID (it could be wrong, or a user could trick it into using someone else's ID).

Instead, pass trusted information as **context** when you call the agent. Tools read it through `ToolRuntime`. The model never sees or controls it.

```
 agent.invoke(messages, context=UserContext(employee_id="E101"))
                                   |
                                   v
           tool reads runtime.context.employee_id   (the model cannot change this)
```

In [ ]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime
import pandas as pd

employees = pd.read_csv("data/employees.csv")

@dataclass
class UserContext:
    employee_id: str

@tool
def my_leave_balance(runtime: ToolRuntime[UserContext]) -> str:
    """Get the leave balance of the employee who is asking."""
    emp_id = runtime.context.employee_id                      # comes from our app, not from the model
    row = employees[employees["employee_id"] == emp_id].iloc[0]
    return f"{row['name']}: {row['annual_leave_balance']} annual leave days, {row['sick_leave_balance']} sick days left."

hr_agent = create_agent(model=llm, tools=[my_leave_balance], context_schema=UserContext,
                        system_prompt="You are an HR assistant. Be brief.")

for emp in ["E101", "E104"]:
    r = hr_agent.invoke({"messages": [{"role": "user", "content": "How many leave days do I have left?"}]},
                        context=UserContext(employee_id=emp))
    print(emp, "->", r["messages"][-1].content)

Same question, different user, different (correct) answer - and the model never had a chance to pick the wrong employee.

## 6. Middleware - customising every step

Middleware lets you plug your own code into the agent loop, without rewriting it. Think of it like security checkpoints at an airport: every passenger (every model call or tool call) passes through them, and each checkpoint can inspect, change or stop things.

```
              before_model         wrap_model_call            after_model
  state --> [ checkpoint ] --> [ checkpoint around MODEL ] --> [ checkpoint ] --> tools --> ...
                                                                         wrap_tool_call wraps each tool
```

| Hook | Runs | Typical use |
|------|------|-------------|
| `@before_model` | before each model call | logging, trimming, guardrails |
| `@after_model` | after each model call | checking the output |
| `@dynamic_prompt` | builds the system prompt per call | personalised prompts |
| `@wrap_tool_call` | around each tool call | logging, error handling, caching |

LangChain also ships ready-made middleware: `SummarizationMiddleware`, `HumanInTheLoopMiddleware`, `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware`, `PIIMiddleware`, `ModelFallbackMiddleware` and more.

### 6.1 A dynamic system prompt

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LearnerContext:
    name: str
    level: str

@dynamic_prompt
def personalised_prompt(request: ModelRequest) -> str:
    ctx = request.runtime.context
    return f"You are a Python tutor for {ctx.name}, who is a {ctx.level}. Answer in 2 sentences."

tutor = create_agent(model=llm, tools=[], middleware=[personalised_prompt], context_schema=LearnerContext)

q = {"messages": [{"role": "user", "content": "What is a decorator?"}]}
print("Beginner:", tutor.invoke(q, context=LearnerContext("Ravi", "complete beginner"))["messages"][-1].content)
print("Expert  :", tutor.invoke(q, context=LearnerContext("Meera", "senior engineer"))["messages"][-1].content)

### 6.2 Logging every step, and a safety limit

Let's write two tiny pieces of middleware: one prints how many messages go into each model call, one logs every tool call and how long it took. We also add a built-in limit so a confused agent can never call the model more than 5 times in one run.

In [ ]:
import time
from langchain.agents.middleware import before_model, wrap_tool_call, ModelCallLimitMiddleware

@before_model
def log_model_call(state, runtime):
    print(f"   -> calling model with {len(state['messages'])} messages")
    return None                                   # returning None means "change nothing"

@wrap_tool_call
def time_tool_call(request, handler):
    start = time.time()
    result = handler(request)                     # actually run the tool
    print(f"   -> tool {request.tool_call['name']} took {time.time() - start:.3f}s")
    return result

logged_agent = create_agent(
    model=llm,
    tools=[calculator, get_weather],
    middleware=[log_model_call, time_tool_call, ModelCallLimitMiddleware(run_limit=5)],
)

out = logged_agent.invoke({"messages": [{"role": "user", "content": "Weather in Mumbai and 45 * 12?"}]})
print("\nAnswer:", out["messages"][-1].content)

### 6.3 Automatic summarisation of long chats

On Day 4 we summarised old messages by hand. `SummarizationMiddleware` does it automatically when a conversation passes a threshold. Here we use a tiny threshold (6 messages) so you can see it happen.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

summary_agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(model=llm, trigger=("messages", 6), keep=("messages", 2))],
)

cfg = {"configurable": {"thread_id": "long_chat"}}
sizes = []
for i, text in enumerate([
    "Hi, I'm Kiran and my favourite language is Python.",
    "I am preparing for the AWS Solutions Architect exam.",
    "Give me one tip for the exam.",
    "Give me another tip.",
    "And one more.",
    "What is my name and which exam am I preparing for?",
]):
    r = summary_agent.invoke({"messages": [{"role": "user", "content": text}]}, cfg)
    sizes.append(len(summary_agent.get_state(cfg).values["messages"]))

print(r["messages"][-1].content)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.plot(range(1, len(sizes) + 1), sizes, marker="o")
plt.xlabel("User turn")
plt.ylabel("Messages stored")
plt.title("Summarisation keeps the history short")
plt.grid(alpha=0.3)
plt.show()

The number of stored messages drops once the threshold is crossed: old messages are replaced by a summary. In a real app you would use a token threshold instead, for example `trigger=("tokens", 4000)`.

## 7. Human-in-the-loop - approve before acting

Some tools should never run without a human saying yes: sending emails, issuing refunds, deleting records. `HumanInTheLoopMiddleware` **pauses** the agent before such a tool runs. Your app shows the request to a person, and then resumes with their decision.

```
 user: "email the team about the outage"
        |
        v
  model decides: send_email(to=..., subject=...)
        |
        v
  PAUSE  ------------>  human reviews  ---- approve / edit / reject ----+
                                                                         |
  resume with Command(resume=...)  <-------------------------------------+
        |
        v
  tool runs (or not)  -->  final answer
```

Pausing needs a checkpointer, because the agent's state must be saved while it waits.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a colleague."""
    return f"Email sent to {to} with subject '{subject}'."

email_agent = create_agent(
    model=llm,
    tools=[send_email],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],   # pause before send_email
)

cfg = {"configurable": {"thread_id": "email_1"}}
paused = email_agent.invoke(
    {"messages": [{"role": "user", "content": "Email it-team@novatech.example that the VPN is down."}]}, cfg
)

In [ ]:
request = paused["__interrupt__"][0].value["action_requests"][0]    # what the agent wants to do
print("The agent wants to run:", request["name"])
print("With arguments        :", request["args"])

Now *you* are the human reviewer. Change `decision` below to `"reject"` and re-run both cells to see the difference.

In [ ]:
decision = "approve"           # try "reject" too

if decision == "approve":
    resumed = email_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg)
else:
    resumed = email_agent.invoke(
        Command(resume={"decisions": [{"type": "reject", "message": "Do not send emails right now."}]}), cfg)

print(resumed["messages"][-1].content)

## Mini project: NovaTech HR agent

Let's bring together the whole course so far into one agent:

- **RAG as a tool** (Day 6): search the policy documents
- **data lookup** (Day 7) with **runtime context** (today): the asking employee's own leave balance
- **calculator** for leave and expense maths
- **memory** across turns
- **logging middleware** so we can watch it work

```
                       +--> search_policies   (RAG over data/policies)
 employee question --> |--> my_leave_balance  (employees.csv, uses context)
       (+ memory)      +--> calculator
```

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

docs = (DirectoryLoader("data/policies", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}).load()
        + PyPDFLoader("data/policies/code_of_conduct.pdf").load())
for d in docs:
    d.metadata["file"] = os.path.basename(d.metadata["source"])

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=60).split_documents(docs)
retriever = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small")).as_retriever(search_kwargs={"k": 3})

@tool
def search_policies(query: str) -> str:
    """Search NovaTech's HR, IT security, travel and conduct policies. Use for any company rule or policy question."""
    found = retriever.invoke(query)
    return "\n\n".join(f"[{d.metadata['file']}] {d.page_content}" for d in found)

print(search_policies.invoke({"query": "work from home"})[:300])

In [ ]:
@wrap_tool_call
def show_tool_use(request, handler):
    print(f"   (using {request.tool_call['name']} with {request.tool_call['args']})")
    return handler(request)

novatech_agent = create_agent(
    model=llm,
    tools=[search_policies, my_leave_balance, calculator],
    system_prompt=(
        "You are NovaTech's friendly HR assistant. "
        "For policy questions always use search_policies and mention the source file. "
        "For the employee's own leave balance use my_leave_balance. "
        "If you cannot find an answer, say so and suggest emailing hr@novatech.example."
    ),
    context_schema=UserContext,
    checkpointer=InMemorySaver(),
    middleware=[show_tool_use],
)

def ask_hr(question, employee_id="E101", thread="hr_demo"):
    print("You:", question)
    out = novatech_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        {"configurable": {"thread_id": thread}},
        context=UserContext(employee_id=employee_id),
    )
    print("HR :", out["messages"][-1].content, "\n")

In [ ]:
ask_hr("How many annual leave days do I have left?")
ask_hr("If I take a 5-day trip next month, how many will remain?")
ask_hr("How far in advance do I need to apply for that leave?")
ask_hr("Can I accept a gift worth INR 3,000 from a vendor?")

Look at how much the agent did on its own: it chose the right tool each time, used the calculator for the subtraction, remembered "that leave" from the previous turn, and pulled policy rules from the documents.

### Your turn

Ask as employee `E104` (Karan, who has only 3 days left) whether he can take a 5-day holiday. Use a new `thread` name so the memory starts fresh.

In [ ]:
ask_hr("Can I take a 5-day holiday next month?", employee_id="E104", thread="karan")

## Common mistakes on Day 8

| What you see | What usually went wrong |
|--------------|-------------------------|
| The agent forgets earlier turns | No `checkpointer`, or a different `thread_id` on each call |
| `ValueError` about missing checkpointer with human-in-the-loop | Interrupts need a checkpointer to save the paused state |
| The agent loops many times | Unclear tool descriptions. Add `ModelCallLimitMiddleware` as a safety net. |
| A tool needs the user ID and the model passes the wrong one | Use `context_schema` + `ToolRuntime` instead of a tool argument |

## Quick recap

- An agent is the model-tools loop; `create_agent(model, tools, system_prompt)` builds it.
- `stream_mode="updates"` shows each step; `"messages"` streams tokens.
- `checkpointer` + `thread_id` = memory across turns.
- `response_format` returns a structured final answer.
- `context_schema` + `ToolRuntime` pass trusted information to tools.
- Middleware hooks into every step: dynamic prompts, logging, limits, summarisation, human approval.

## Practice before Day 9

1. Add a tool `company_holidays()` that returns a fixed list, and ask the HR agent about the next holiday.
2. Add `HumanInTheLoopMiddleware` so `calculator` needs approval, and try both approve and reject.
3. Write an `@after_model` middleware that prints the token usage of every model call.
4. Give the HR agent a `response_format` with fields `answer`, `sources`, `needs_hr_followup`.

**Tomorrow:** the things that turn a demo into a product - async calls, caching, cost tracking, callbacks, rate limits, guardrails and serving your chain as an API.